# Model Training Notebook — Beginner Friendly Version

This notebook trains a machine learning model that predicts `salary_in_usd`.

The main idea is simple:

1. Load the cleaned salary data.
2. Choose the columns we want to use as input.
3. Choose the column we want to predict.
4. Split the data into training rows and testing rows.
5. Train a few models.
6. Compare the models.
7. Save the best model so the API can use it later.


## 1. Import the libraries

These libraries help us:

- read the CSV file
- split the data
- prepare text columns for machine learning
- train models
- check how good the models are
- save the final model


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

# This number makes the results repeatable.
# If we run the notebook again, we should get the same split and similar results.
RANDOM_STATE = 42

# Show all columns when we print a DataFrame.
pd.set_option("display.max_columns", None)


## 2. Load the cleaned data

This notebook does not clean the raw dataset.

It uses the file created by the first notebook:

`../data/processed/cleaned_salaries.csv`


In [2]:
data_path = Path("../data/processed/cleaned_salaries.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)

df.head()


Dataset loaded successfully.
Rows and columns: (600, 10)


,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,work_setting,company_location,company_size,salary_in_usd
0,2020,Mid-level,Full-time,Data Scientist,DE,0,On-site,DE,Large,79833
1,2020,Senior-level,Full-time,Machine Learning Scientist,JP,0,On-site,JP,Small,260000
2,2020,Senior-level,Full-time,Big Data Engineer,GB,50,Hybrid,GB,Medium,109024
3,2020,Mid-level,Full-time,Other,HN,0,On-site,HN,Small,20000
4,2020,Senior-level,Full-time,Machine Learning Engineer,US,50,Hybrid,US,Large,150000


## 3. Look at the target column

The target is the value we want the model to predict.

In this project, the target is:

`salary_in_usd`


In [3]:
df["salary_in_usd"].describe()


count       600.000000
mean     108348.011667
std       60763.132982
min        2859.000000
25%       62650.500000
50%      100000.000000
75%      150000.000000
max      380000.000000
Name: salary_in_usd, dtype: float64

## 4. Choose input columns and target column

The model needs:

- **X**: the input columns
- **y**: the answer column we want to predict

Example:

If we give the model job title, experience level, company size, and location,  
it tries to predict the salary.


In [4]:
target_column = "salary_in_usd"

feature_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "remote_ratio",
    "company_location",
    "company_size",
]

X = df[feature_columns].copy()
y = df[target_column].copy()

print("Input columns:")
print(feature_columns)

print("\nTarget column:")
print(target_column)


Input columns:
['work_year', 'experience_level', 'employment_type', 'job_title', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']

Target column:
salary_in_usd


## 5. Separate numeric columns and text columns

Machine learning models work better with numbers.

Some columns are already numbers, like:

- `work_year`
- `remote_ratio`

Other columns are text, like:

- `job_title`
- `experience_level`
- `company_size`

We will convert text columns into numbers later using **one-hot encoding**.


In [5]:
numeric_features = [
    "work_year",
    "remote_ratio",
]

categorical_features = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size",
]

print("Numeric columns:", numeric_features)
print("Text/category columns:", categorical_features)


Numeric columns: ['work_year', 'remote_ratio']
Text/category columns: ['experience_level', 'employment_type', 'job_title', 'employee_residence', 'company_location', 'company_size']


## 6. Split the data into train and test

We do not train the model on all rows.

We split the data into:

- **Training data**: used to teach the model
- **Testing data**: used to check if the model learned well

Here, 80% of the data is for training and 20% is for testing.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


Training rows: 480
Testing rows: 120


## 7. Create a simple evaluation function

After a model makes predictions, we need to measure how good the predictions are.

We will use:

- **MAE**: average error in dollars. Lower is better.
- **RMSE**: gives bigger punishment to large errors. Lower is better.
- **R2**: shows how much the model explains the data. Higher is better.


In [7]:
def calculate_scores(real_values, predicted_values):
    mae = mean_absolute_error(real_values, predicted_values)
    rmse = np.sqrt(mean_squared_error(real_values, predicted_values))
    r2 = r2_score(real_values, predicted_values)

    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }


## 8. Create a baseline

A baseline is a very simple prediction.

Here, the baseline always predicts the median salary from the training data.

A real model should be better than this simple baseline.


In [8]:
baseline_salary = float(y_train.median())

baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=baseline_salary,
)

baseline_scores = calculate_scores(y_test, baseline_predictions)

print(f"Baseline predicted salary: ${baseline_salary:,.2f}")
print(baseline_scores)


Baseline predicted salary: $100,000.00
{'mae': 45745.98333333333, 'rmse': 56916.21563623147, 'r2': -0.016492180186596483}


## 9. Prepare the columns for machine learning

The model cannot understand text directly.

So we use `OneHotEncoder`.

Example:

`company_size = Medium`

becomes a numeric column like:

`company_size_Medium = 1`

This is a common way to convert text categories into numbers.


In [9]:
# Different scikit-learn versions use different names:
# newer versions use sparse_output
# older versions use sparse
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", one_hot_encoder, categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)


## 10. Create the models we want to test

We will test three models:

1. **Decision Tree**
2. **Random Forest**
3. **Gradient Boosting**

All of them are regression models, which means they predict a number.


In [10]:
models_to_try = [
    (
        "Decision Tree",
        DecisionTreeRegressor(
            max_depth=7,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
        ),
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
    (
        "Gradient Boosting",
        GradientBoostingRegressor(
            n_estimators=250,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
    ),
]

for model_name, model in models_to_try:
    print(model_name)


Decision Tree
Random Forest
Gradient Boosting


## 11. Train and test each model

For each model, we will:

1. Create a pipeline.
2. Train it using the training data.
3. Make predictions using the testing data.
4. Save the scores.


In [11]:
results = []
trained_models = {}

for model_name, model in models_to_try:
    # A pipeline connects preprocessing and the model together.
    # This means the same steps will happen every time we train or predict.
    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    # Train the model.
    model_pipeline.fit(X_train, y_train)

    # Predict salaries for the test rows.
    predictions = model_pipeline.predict(X_test)

    # Calculate the model scores.
    scores = calculate_scores(y_test, predictions)

    # Save the trained model so we can use it later.
    trained_models[model_name] = model_pipeline

    # Save the results in a list.
    results.append(
        {
            "candidate": model_name,
            "model_name": model_name,
            "target_transform": "none",
            "mae": scores["mae"],
            "rmse": scores["rmse"],
            "r2": scores["r2"],
        }
    )

comparison_df = pd.DataFrame(results)

# Best model = lowest MAE.
# If two models have similar MAE, higher R2 is better.
comparison_df = comparison_df.sort_values(
    by=["mae", "r2"],
    ascending=[True, False],
).reset_index(drop=True)

comparison_df


,candidate,model_name,target_transform,mae,rmse,r2
0,Random Forest,Random Forest,none,25751.590839,34480.889984,0.626931
1,Gradient Boosting,Gradient Boosting,none,26200.641796,34017.241453,0.636897
2,Decision Tree,Decision Tree,none,28413.386523,37102.180311,0.568053


## 12. Choose the best model

The first row in `comparison_df` is the best model based on MAE.

MAE is easy to understand because it is measured in dollars.


In [12]:
best_row = comparison_df.iloc[0]

best_model_name = best_row["candidate"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

print("\nBest model scores:")
print({
    "mae": best_row["mae"],
    "rmse": best_row["rmse"],
    "r2": best_row["r2"],
})

print("\nBaseline scores:")
print(baseline_scores)

mae_improvement = baseline_scores["mae"] - best_row["mae"]
r2_improvement = best_row["r2"] - baseline_scores["r2"]

print(f"\nMAE improvement compared to baseline: ${mae_improvement:,.2f}")
print(f"R2 improvement compared to baseline: {r2_improvement:.4f}")


Best model: Random Forest

Best model scores:
{'mae': np.float64(25751.590839141627), 'rmse': np.float64(34480.889983815956), 'r2': np.float64(0.6269311886086848)}

Baseline scores:
{'mae': 45745.98333333333, 'rmse': 56916.21563623147, 'r2': -0.016492180186596483}

MAE improvement compared to baseline: $19,994.39
R2 improvement compared to baseline: 0.6434


## 13. Try one sample prediction

Now we create one example row and ask the model to predict the salary.

The input must use the same columns used during training.


In [13]:
sample_input = pd.DataFrame(
    [
        {
            "work_year": 2022,
            "experience_level": "Senior-level",
            "employment_type": "Full-time",
            "job_title": "Data Scientist",
            "employee_residence": "US",
            "remote_ratio": 100,
            "company_location": "US",
            "company_size": "Medium",
        }
    ]
)

sample_prediction = best_model.predict(sample_input)[0]

print(f"Sample predicted salary: ${sample_prediction:,.2f}")


Sample predicted salary: $165,924.42


## 14. Save the best model

The API needs the saved model file so it can make predictions without retraining the model every time.


In [14]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "salary_best_model_pipeline.joblib"

joblib.dump(best_model, model_path)

print("Model saved to:")
print(model_path)


Model saved to:
..\models\salary_best_model_pipeline.joblib


## 15. Save the model scores

We save the scores in a JSON file.

This makes it easy to show the model performance later in the API, dashboard, or documentation.


In [15]:
metrics = {
    "model_name": best_row["model_name"],
    "candidate": best_model_name,
    "target": target_column,
    "features": feature_columns,
    "target_transform": best_row["target_transform"],
    "mae": float(best_row["mae"]),
    "rmse": float(best_row["rmse"]),
    "r2": float(best_row["r2"]),
    "baseline_strategy": "median_training_salary",
    "baseline_salary": baseline_salary,
    "baseline_mae": baseline_scores["mae"],
    "baseline_rmse": baseline_scores["rmse"],
    "baseline_r2": baseline_scores["r2"],
    "test_size": 0.2,
    "random_state": RANDOM_STATE,
    "model_comparison": comparison_df.to_dict(orient="records"),
}

metrics_path = models_dir / "model_metrics.json"

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Metrics saved to:")
print(metrics_path)


Metrics saved to:
..\models\model_metrics.json


## 16. Save the allowed input values

The API and frontend need to know which values are allowed.

For example, the frontend can use this file to create dropdowns for:

- job titles
- experience levels
- company sizes
- countries


In [16]:
input_schema = {
    "work_year": sorted(int(value) for value in X["work_year"].dropna().unique()),
    "experience_level": sorted(X["experience_level"].dropna().unique().tolist()),
    "employment_type": sorted(X["employment_type"].dropna().unique().tolist()),
    "job_title": sorted(X["job_title"].dropna().unique().tolist()),
    "employee_residence": sorted(X["employee_residence"].dropna().unique().tolist()),
    "remote_ratio": sorted(int(value) for value in X["remote_ratio"].dropna().unique()),
    "company_location": sorted(X["company_location"].dropna().unique().tolist()),
    "company_size": sorted(X["company_size"].dropna().unique().tolist()),
}

schema_path = models_dir / "input_schema.json"

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(input_schema, file, indent=4)

print("Input schema saved to:")
print(schema_path)


api_dir = Path("../api")
api_dir.mkdir(parents=True, exist_ok=True)

api_allowed_values_path = api_dir / "allowed_values.json"

with open(api_allowed_values_path, "w", encoding="utf-8") as file:
    json.dump(input_schema, file, indent=4)

print("\nAPI allowed values saved to:")
print(api_allowed_values_path)


Input schema saved to:
..\models\input_schema.json

API allowed values saved to:
..\api\allowed_values.json


## 17. Save feature importance

Feature importance helps us understand which columns affected the model most.

This only works for models that support `feature_importances_`.

The selected model here is tree-based, so it should usually support this.


In [17]:
feature_importance_path = Path("../data/processed/feature_importance.csv")

final_preprocessor = best_model.named_steps["preprocessor"]
final_model = best_model.named_steps["model"]

if hasattr(final_model, "feature_importances_"):
    feature_names = final_preprocessor.get_feature_names_out()

    feature_importance_df = pd.DataFrame(
        {
            "feature": feature_names,
            "importance": final_model.feature_importances_,
        }
    )

    feature_importance_df = feature_importance_df.sort_values(
        by="importance",
        ascending=False,
    )

    feature_importance_df.to_csv(feature_importance_path, index=False)

    print("Feature importance saved to:")
    print(feature_importance_path)

    feature_importance_df.head(20)
else:
    print("This model does not support feature importance.")

    feature_importance_df = pd.DataFrame(
        columns=["feature", "importance"]
    )

    feature_importance_df.to_csv(feature_importance_path, index=False)


Feature importance saved to:
..\data\processed\feature_importance.csv


## Notebook summary

In this notebook, we:

1. Loaded the cleaned salary dataset.
2. Selected input columns and the target column.
3. Split the data into training and testing data.
4. Created a simple baseline.
5. Trained three regression models.
6. Selected the best model.
7. Tested one sample prediction.
8. Saved the model, metrics, input schema, and feature importance.
